# Go Game Review Analyser

Date: `2026-03-25`

Author: `@sthornewillve`

This notebook is the prototype that takes go games analysed and creates a database from them in order to analyse.

___

# Setup Notebook

In [1]:
import os
import json
import pandas as pd
import sqlite3

from openai import OpenAI
from go_game_analyser_helperfuncs import parse_game_reviews, summarise_game_reviews

In [2]:
# Create OpenAI Connection
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Import prompts
with open("prompt_configs.json") as f:
    prompts = json.load(f)

In [3]:
# Get game reviews
game_review_data, game_review_df = parse_game_reviews()

game_review_df.head()

100%|██████████| 4/4 [00:00<00:00, 3058.74it/s]


,date,opponents_name,server,game_link,result,played_as,handicap,time_setting,review_notes
0,2026-03-24,dienfoonwu,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+R,White,3,20m + 5x30s,\n* End of game notes\n\t* Got surrounded and ...
1,2026-03-23,Yaryk1111_,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+2.5,Black,0,20m + 5x30s,\n* End of game notes\n\t* Black has a large l...
2,2026-03-24,njbigboy,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,W+T,White,1,10m + 10s/mv,\n* End of game notes\n\t* Died with a deep in...
3,2026-03-25,이송희,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,W+5.5,Black,0,20m + 5x30s,\n* End of game notes\n\t* Black and white tra...


___

# Analyse Games

In [4]:
game_summaries = summarise_game_reviews(game_review_data, client, prompts)

100%|██████████| 4/4 [01:17<00:00, 19.49s/it]


In [5]:
for aspect in ["key_mistake", "key_mistake_cause", "positive_point", "game_tags"]:
    game_review_df[aspect] = [i[aspect] for i in game_summaries.values()]

game_review_df

,date,opponents_name,server,game_link,result,played_as,handicap,time_setting,review_notes,key_mistake,key_mistake_cause,positive_point,game_tags
0,2026-03-24,dienfoonwu,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+R,White,3,20m + 5x30s,\n* End of game notes\n\t* Got surrounded and ...,Letting atari surround my group,Poor defense against atari,Exploited opponent's weak shape,Allowed group to get surrounded; Endgame troub...
1,2026-03-23,Yaryk1111_,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,B+2.5,Black,0,20m + 5x30s,\n* End of game notes\n\t* Black has a large l...,Attachment to shimari as ladder breaker,tried to salvage with attachment instead of co...,Handled the running battle better than White,Exploited opponent's weak shape; Ko situation;...
2,2026-03-24,njbigboy,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,W+T,White,1,10m + 10s/mv,\n* End of game notes\n\t* Died with a deep in...,Died with a deep invasion,abandoning deep invasion,Nice framework on the left side,Opponent ran out of time; Played calmly; Built...
3,2026-03-25,이송희,OGS,https://ai-sensei.com/game/VGIPT5au1Jg6jxQRo6P...,W+5.5,Black,0,20m + 5x30s,\n* End of game notes\n\t* Black and white tra...,Ending in gote,Endgame misjudgment leading to gote,Captured White's center group in the opening,Lost points in endgame; Endgame trouble; Compl...


In [ ]:
conn = sqlite3.connect("game_reviews.db")
game_review_df.to_sql("reviews", conn, if_exists="replace", index=False)  # TODO: Make it so that only new games are analysed


4


___
# Appendix
## Using OpenAI API

Note that `messages` should be in the form of a list of dictionaries that has the keys `'role'` and `'content'`. Where `'role'` can be either...

1. `'system'`: The system prompt, defining the task and role for the LLM.
2. `'user'`: The actual input that you want it to process.

Be sure to be very specific about the output, everything that's not specified will be done randomly. It's usually a good idea to verify the output of the LLM and try again if it doesn't work. 

This is an example dictionary to use...

```python
messages = [
    {"role": "system", 
     "content": "You are a.../n\
     \
     Input: State input.../n\
     \
     Task: State task.../n\
     \
     Output: The Exact output..."},
    
    {"role": "user", 
     "content": "Input 1:\n...\n\n\
     \
     Input 21:\n...\n\n\
     \
     Return as described in the system message."}
]
```